# Silver Layer — ERP Product Category (PX_CAT_G1V2)
Clean and normalize `erp_px_cat_g1v2`.

## Setup Connection

In [ ]:
import os
from dotenv import load_dotenv
from clickzetta.zettapark.session import Session

load_dotenv()
session = Session.builder.configs({
    "username":  os.environ["CLICKZETTA_USERNAME"],
    "password":  os.environ["CLICKZETTA_PASSWORD"],
    "service":   os.environ["CLICKZETTA_SERVICE"],
    "instance":  os.environ["CLICKZETTA_INSTANCE"],
    "workspace": os.environ["CLICKZETTA_WORKSPACE"],
    "schema":    os.environ["CLICKZETTA_SCHEMA"],
    "vcluster":  os.environ["CLICKZETTA_VCLUSTER"],
}).create()
SCHEMA = os.environ["CLICKZETTA_SCHEMA"]

## Read Bronze Table

In [ ]:
df = session.table(f"{SCHEMA}.erp_px_cat_g1v2")

## Silver Transformations

### Trimming

In [ ]:
from clickzetta.zettapark.types import StringType
from clickzetta.zettapark import functions as F

for field in df.schema.fields:
    if isinstance(field.datatype, StringType):
        df = df.with_column(field.name, F.trim(F.col(field.name)))

### Maintenance Flag Normalization
Convert YES/NO string to boolean.

In [ ]:
df = df.with_column(
    "maintenance",
    F.when(F.upper(F.col("maintenance")) == "YES", F.lit(True))
     .when(F.upper(F.col("maintenance")) == "NO", F.lit(False))
     .otherwise(F.lit(None))
)

### Rename Columns

In [ ]:
RENAME_MAP = {
    "id":          "category_id",
    "cat":         "category",
    "subcat":      "subcategory",
    "maintenance": "maintenance_flag",
}
for old, new in RENAME_MAP.items():
    df = df.with_column_renamed(old, new)

## Sanity Check

In [ ]:
df.limit(10).show()

## Write Silver Table

In [ ]:
df.write.save_as_table(f"{SCHEMA}.erp_product_category", mode="overwrite")
print("erp_product_category OK")

## Verify

In [ ]:
session.table(f"{SCHEMA}.erp_product_category").limit(5).show()